### ライブラリのインポート

In [ ]:
import onnxruntime as ort

print(ort.__version__)
print(ort.get_available_providers())

sess = ort.InferenceSession(
    "./onnx/fastbev.onnx",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)
print("ok")

In [5]:
import numpy as np
import torch
import onnxruntime as ort
import torch.nn.functional as F

from nuscenes_sample_generator import NuScenesSampleGenerator
from img_pipeline import PrepareImageInputs
from onnx_input_builder import OnnxInputBuilder
from bbox_decoder import BboxDecoder
from visualizer import Visualizer 
from util import load_config, load_json

### 定数設定

In [6]:
# パスの設定
json_path   = "bevdetv3-nuscenes_infos_val.json"
config_path = "config.yaml"

### 初期化処理

In [3]:
# jsonファイルとconfigファイルの読み込み
json_file   = load_json(json_path)
data_config = load_config(config_path)

# sample Generator
data_infos = json_file['infos']
sample_generator = NuScenesSampleGenerator(data_infos=data_infos, num_adj_frame=1)

# image pipeline
image_pipline = PrepareImageInputs(data_config["data_config"], sequential=True, opencv_pp=False)

# 入力データの作成
grid_config = data_config["geometry"]["grid_config"]
image_size  = data_config["data_config"]["input_size"]
onnx_input_builder = OnnxInputBuilder(grid_config, image_size, stride=16, accelerate=True)

# モデルの後処理
post_center_range = data_config["bbox_decoder"]["post_center_range"]
max_num           = data_config["bbox_decoder"]["max_num"]
num_classes       = data_config["bbox_decoder"]["num_classes"]
score_threshold   = data_config["bbox_decoder"]["score_threshold"]
bbox_decoder = BboxDecoder(post_center_range, max_num, num_classes, None)

# 可視化
save_path   = data_config["visualizer"]["save_path"]
save_format = data_config["visualizer"]["save_format"]
save_prefix = data_config["visualizer"]["save_prefix"]
fps         = data_config["visualizer"]["fps"]
visualizer  = Visualizer(save_path, save_format, save_prefix, fps, scale_factor=3, color_map=(0, 255, 255))


# モデルの読み込み
fastbev = ort.InferenceSession(
    "./onnx/fastbev.onnx",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
    #providers=["CPUExecutionProvider"]
)

fastbev4d = ort.InferenceSession(
    "./onnx/fastbev_4d.onnx",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
    #providers=["CPUExecutionProvider"]
)

print("===fastbev===")
print("=== Inputs ===")
for i, x in enumerate(fastbev.get_inputs()):
    print(f"[{i}]")
    print(" name :", x.name)
    print(" shape:", x.shape)
    print(" type :", x.type)

print("\n=== Outputs ===")
for i, x in enumerate(fastbev.get_outputs()):
    print(f"[{i}]")
    print(" name :", x.name)
    print(" shape:", x.shape)
    print(" type :", x.type)

print("\n===fastbev4d===")
print("=== Inputs ===")
for i, x in enumerate(fastbev4d.get_inputs()):
    print(f"[{i}]")
    print(" name :", x.name)
    print(" shape:", x.shape)
    print(" type :", x.type)

print("\n=== Outputs ===")
for i, x in enumerate(fastbev4d.get_outputs()):
    print(f"[{i}]")
    print(" name :", x.name)
    print(" shape:", x.shape)
    print(" type :", x.type)

# warmup



NameError: name 'load_json' is not defined

### １サンプルでの確認

# 前処理

In [2]:
data_infos[0]["token"]

NameError: name 'data_infos' is not defined

In [14]:
# 1サンプルでの確認
input_data = sample_generator.get_data_info(0)

# 前処理
input_data = image_pipline(input_data)
imgs_curr, sensor2keyegos_curr, ego2globals_curr, intrins_curr, post_rots_curr, post_trans_curr, bda_curr = input_data["img_inputs_curr"]

print(sensor2keyegos_curr)
print(ego2globals_curr)

# fastbev4dの確認
#data_origin = torch.load("../debug_dump/data_cpu.pt", weights_only=False)

tensor([[[[ 0.8225,  0.0065,  0.5687,  1.5753],
          [-0.5687,  0.0164,  0.8224,  0.5005],
          [-0.0040, -0.9998,  0.0172,  1.5070],
          [ 0.0000,  0.0000,  0.0000,  1.0000]],

         [[ 0.0100,  0.0085,  0.9999,  1.7888],
          [-0.9999,  0.0122,  0.0099,  0.0041],
          [-0.0121, -0.9999,  0.0086,  1.4962],
          [ 0.0000,  0.0000,  0.0000,  1.0000]],

         [[-0.8442,  0.0166,  0.5357,  1.7182],
          [-0.5358,  0.0035, -0.8444, -0.5004],
          [-0.0159, -0.9999,  0.0060,  1.5202],
          [ 0.0000,  0.0000,  0.0000,  1.0000]],

         [[ 0.9483, -0.0082, -0.3174,  1.4223],
          [ 0.3175,  0.0180,  0.9481,  0.4805],
          [-0.0020, -0.9998,  0.0197,  1.5691],
          [ 0.0000,  0.0000,  0.0000,  1.0000]],

         [[ 0.0101, -0.0062, -0.9999,  0.3424],
          [ 0.9999,  0.0107,  0.0101,  0.0097],
          [ 0.0107, -0.9999,  0.0064,  1.5731],
          [ 0.0000,  0.0000,  0.0000,  1.0000]],

         [[-0.9235, -0.0023, -

In [16]:
bda_curr

tensor([[[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.],
         [0., 0., 0., 1.]]])

In [8]:
# 1サンプルでの確認
input_data = sample_generator.get_data_info(0)

# 前処理
input_data = image_pipline(input_data)

# FastBEVモデルの入力データ作成
# 現在フレームの入力データ
img_curr = input_data["img_inputs_curr"][0]
_, coors_img_curr, coors_depth_curr = onnx_input_builder.get_fastray_input(input_data["img_inputs_curr"]) # 事前計算済みの座標変換情報
coors_img_curr, coors_depth_curr = coors_img_curr[0], coors_depth_curr[0]
# tensor→numpyに変換(ORTにはnumpyで渡す)
img_curr_np = img_curr.squeeze(0).detach().cpu().numpy().astype(np.float32)
coors_img_curr_np = coors_img_curr.detach().cpu().numpy().astype(np.int64)
coors_depth_curr_np = coors_depth_curr.detach().cpu().numpy().astype(np.int64)
#print(type(img_curr), img_curr.dtype, img_curr.shape)
#print(type(coors_img_curr), coors_img_curr.dtype, coors_img_curr.shape)

# 過去フレームの入力データ
img_prev = input_data["img_inputs_prev"][0]
_, coors_img_prev, coors_depth_prev = onnx_input_builder.get_fastray_input(input_data["img_inputs_prev"]) # 事前計算済みの座標変換情報
coors_img_prev, coors_depth_prev = coors_img_prev[0], coors_depth_prev[0]
# tensor→numpyに変換(ORTにはnumpyで渡す)
img_prev_np = img_prev.squeeze(0).detach().cpu().numpy().astype(np.float32)
coors_img_prev_np = coors_img_prev.detach().cpu().numpy().astype(np.int64)
coors_depth_prev_np = coors_depth_prev.detach().cpu().numpy().astype(np.int64)

# BEV特徴量の作成
bev_feat_curr = fastbev.run(
    ["bev_feat"],
    {
        "img": img_curr_np,
        "coors_img": coors_img_curr_np,
        "coors_depth": coors_depth_curr_np,
    }
)[0]

print(bev_feat_curr)

bev_feat_prev = fastbev.run(
    ["bev_feat"],
    {
        "img": img_prev_np,
        "coors_img": coors_img_prev_np,
        "coors_depth": coors_depth_prev_np,
    }
)[0]

# 後段の処理のために、一旦torchに戻す
bev_feat_curr = torch.from_numpy(bev_feat_curr).float()
bev_feat_prev = torch.from_numpy(bev_feat_prev).float()

# FastBEV4Dモデルへの入力データ作成
# 過去のBEV特徴量を現在のBEV特徴量に整列させる
#_, sensor2keyegos_curr, _, _, _, _, bda_curr = input_data["img_inputs_curr"]
#_, sensor2keyegos_prev, _, _, _, _, bda_prev = input_data["img_inputs_prev"]
#_, C, H, W = bev_feat_prev.shape
#grid = onnx_input_builder.gen_grid(bev_feat_prev, [sensor2keyegos_curr, sensor2keyegos_prev], bda_curr)
#bev_feat_prev_align = F.grid_sample(bev_feat_prev, grid.to(bev_feat_prev.dtype), align_corners=True)
# 過去と現在フレームのBEVを結合
#num_frame = 2
#bev_feats_np = torch.cat(
#    [
#        bev_feat_curr,
#        bev_feat_prev.view(1, (num_frame - 1) * C, H, W)
#    ],
#    dim=1
#).detach().cpu().numpy().astype(np.float32)

# 物体認識の実行
#outputs = fastbev4d.run(
#    None,
#    {"bev_feat": bev_feats_np}
#)

#all_cls_scores = outputs[0]
#all_bbox_preds = outputs[1]

#outputs = [torch.from_numpy(x).float() for x in outputs]

# 後処理
#bboxes, scores, labels = bbox_decoder.get_bbox(outputs)

# 可視化
#drawed_img = visualizer.draw_bbox(data_infos[1], bboxes, 1)



[[[[0.361174   0.8232275  0.69101167 ... 0.95533764 0.9273912
    0.7222649 ]
   [0.5275859  0.8867201  0.613609   ... 1.1407835  1.10486
    0.67924535]
   [0.52221876 0.8735046  0.5467324  ... 1.1541541  1.1158066
    0.68109   ]
   ...
   [0.27782857 1.6419435  1.0809686  ... 1.1311109  1.1051617
    0.6730943 ]
   [0.12069365 0.9778656  1.0630176  ... 1.0926929  1.0818725
    0.67102695]
   [0.2124684  0.6090296  0.4281075  ... 0.43463957 0.43611214
    0.0735223 ]]

  [[0.         0.01074165 0.         ... 0.15672827 0.15042377
    0.        ]
   [0.         0.         0.         ... 0.         0.
    0.        ]
   [0.         0.         0.         ... 0.         0.
    0.        ]
   ...
   [0.         0.16202271 0.29035908 ... 0.         0.
    0.        ]
   [0.         0.         0.         ... 0.         0.
    0.        ]
   [0.         0.         0.         ... 0.         0.
    0.        ]]

  [[0.6853427  0.67597705 1.3262709  ... 0.56774014 0.53873855
    0.35712594]
  

In [9]:
print(bev_feat_curr)

tensor([[[[0.3612, 0.8232, 0.6910,  ..., 0.9553, 0.9274, 0.7223],
          [0.5276, 0.8867, 0.6136,  ..., 1.1408, 1.1049, 0.6792],
          [0.5222, 0.8735, 0.5467,  ..., 1.1542, 1.1158, 0.6811],
          ...,
          [0.2778, 1.6419, 1.0810,  ..., 1.1311, 1.1052, 0.6731],
          [0.1207, 0.9779, 1.0630,  ..., 1.0927, 1.0819, 0.6710],
          [0.2125, 0.6090, 0.4281,  ..., 0.4346, 0.4361, 0.0735]],

         [[0.0000, 0.0107, 0.0000,  ..., 0.1567, 0.1504, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          ...,
          [0.0000, 0.1620, 0.2904,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],

         [[0.6853, 0.6760, 1.3263,  ..., 0.5677, 0.5387, 0.3571],
          [0.2363, 0.0352, 0.1746,  ..., 0.0000, 0.0000, 0.0000],
          [0.1791, 0.0255, 0.2739,  ..., 0

In [7]:
print(coors_img_curr)

tensor([3081, 3037, 3037,  ...,    0,    0,    0])


In [6]:
outputs = [torch.from_numpy(x).float() for x in outputs]

# 後処理
bboxes, scores, labels = bbox_decoder.get_bbox(outputs)

# 可視化
drawed_img = visualizer.draw_bbox(data_infos[1], bboxes, 1)

/home/kenta/fastbev/model_deploy/visualizer.py:103: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  bboxes_tensor = torch.tensor(bboxes)


In [8]:
print(type(bev_feat_prev))

<class 'numpy.ndarray'>


In [9]:
type(outputs)
outputs[0]

array([[[-1.4260104, -2.367527 , -4.790111 , ..., -4.657922 ,
         -2.8775027, -3.8359   ],
        [-1.4317055, -2.33947  , -4.7084537, ..., -4.6685915,
         -2.8853803, -3.7909107],
        [-1.5225422, -2.4050655, -4.755465 , ..., -5.1083255,
         -3.0519414, -3.845858 ],
        ...,
        [-1.4139993, -2.3756125, -5.010224 , ..., -4.7847066,
         -2.9822524, -3.8918252],
        [-1.4138641, -2.3579435, -4.901739 , ..., -4.8561435,
         -3.0105946, -3.8045974],
        [-1.4533627, -2.380096 , -4.8256216, ..., -4.7949758,
         -2.9725583, -3.780339 ]]], shape=(1, 300, 10), dtype=float32)

In [8]:
coors_img_curr

[tensor([725121, 724417, 724417,  ...,      0,      0,      0])]

In [14]:
input_data["img_inputs"][0].shape

torch.Size([12, 3, 256, 704])

In [6]:
print(input_data.keys())

dict_keys(['sample_idx', 'timestamp', 'curr', 'adjacent', 'cam_names', 'canvas', 'img_inputs'])


In [10]:
input_data['img_inputs']

(tensor([[[[ 0.2282,  0.2282,  0.2282,  ..., -1.6042, -1.5699, -1.5185],
           [ 0.1939,  0.1768,  0.1768,  ..., -1.6042, -1.5185, -1.5014],
           [ 0.1768,  0.1597,  0.1597,  ..., -1.5185, -1.4500, -1.5528],
           ...,
           [-0.8507, -0.7993, -0.7650,  ..., -1.0048, -1.0390, -1.0562],
           [-0.7822, -0.7479, -0.7137,  ..., -0.9363, -0.9534, -0.9705],
           [-0.7993, -0.7650, -0.7650,  ..., -0.9705, -1.0219, -0.9705]],
 
          [[ 0.1877,  0.2052,  0.2052,  ..., -1.4055, -1.2479, -1.1429],
           [ 0.1877,  0.1702,  0.1702,  ..., -1.3004, -1.2129, -1.1779],
           [ 0.1702,  0.1527,  0.1527,  ..., -1.2304, -1.1429, -1.2304],
           ...,
           [-0.7052, -0.6527, -0.6176,  ..., -0.7577, -0.7927, -0.8102],
           [-0.6001, -0.5651, -0.5476,  ..., -0.7402, -0.7227, -0.7402],
           [-0.6176, -0.5826, -0.5826,  ..., -0.7927, -0.7927, -0.7402]],
 
          [[ 0.2871,  0.3045,  0.3045,  ..., -1.2119, -1.1247, -1.0201],
           [ 

In [14]:
rotate_angle = 0
scale_ratio = 1.0
flip_dx = False
flip_dy = False
tran_bda = np.zeros((1, 3), dtype=np.float32)

# BEVオーグメンテーションの4×4の同次行列の作成
bda_mat = torch.zeros(4, 4)
bda_mat[3, 3] = 1

# 回転行列の作成
rotate_angle = torch.tensor(rotate_angle / 180 * np.pi) # 回転角度をラジアンに変換
rot_sin = torch.sin(rotate_angle)
rot_cos = torch.cos(rotate_angle)
rot_mat = torch.Tensor([[rot_cos, -rot_sin, 0], 
                        [rot_sin,  rot_cos, 0],
                        [0,              0, 1]])

# スケール行列の作成
scale_mat = torch.Tensor([[scale_ratio,           0,           0], 
                          [0,           scale_ratio,           0],
                          [0,                     0, scale_ratio]])

# 反転行列の作成
flip_mat = torch.Tensor([[1, 0, 0], 
                         [0, 1, 0], 
                         [0, 0, 1]])

if flip_dx: # 水平反転
    flip_mat = flip_mat @ torch.Tensor([[-1, 0, 0], 
                                        [0, 1, 0],
                                        [0, 0, 1]])
if flip_dy: # 垂直反転
    flip_mat = flip_mat @ torch.Tensor([[1, 0, 0], 
                                        [0, -1, 0],
                                        [0, 0, 1]])

# 回転・拡大縮小・反転を合成
bda_rot = flip_mat @ (scale_mat @ rot_mat)

# 同次行列に変換行列を保存
bda_mat[:3, :3] = bda_rot
bda_mat[:3, 3] = torch.from_numpy(tran_bda)

print(bda_mat)

tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]])


In [16]:
def sample_augmentation(H, W, flip=None, scale=None):
    """画像のリサイズ・クロップ・反転・回転の augmentation パラメータを決定する。

    学習時はランダムaugmentationをサンプリングし、テスト時は決定的な前処理パラメータを返す。

    Args:
        H (int): 元画像の高さ。
        W (int): 元画像の幅。
        flip (bool, optional): テスト時に明示的に左右反転を指定するかどうか。デフォルトは None。
        scale (float, optional): テスト時に追加のリサイズ量を指定する値。デフォルトは None。

    Returns:
        tuple:
            - resize (float): リサイズ倍率
            - resize_dims (tuple[int, int]): リサイズ後の画像サイズ (W, H)
            - crop (tuple[int, int, int, int]): クロップ領域 (left, top, right, bottom)
            - flip (bool): 左右反転するかどうか
            - rotate (float): 回転角度
    """

    # モデルへの最終的な入力画像サイズを取得
    fH, fW = [256, 704]

    # オーグメンテーションの適用
    # アスペクト比を保ってリサイズ→必要な領域をクロップする(学習時には余分に大きくリサイズしてクロップする)
    # リサイズ率の設定
    resize = float(fW) / float(W)
    if scale is not None:
        resize += scale
    else:
        resize += 0.0
    # リサイズ後の画像サイズの決定
    resize_dims = (int(W * resize), int(H * resize))
    newW, newH = resize_dims
    # クロップ領域の設定
    crop_h = int((1 - np.mean([0.0, 0.0])) * newH) - fH
    crop_w = int(max(0, newW - fW) / 2)
    crop = (crop_w, crop_h, crop_w + fW, crop_h + fH)
    # フリップの設定
    flip = False if flip is None else flip
    # 回転の設定
    rotate = 0
    
    return resize, resize_dims, crop, flip, rotate

In [17]:
resize, resize_dims, crop, flip, rotate = sample_augmentation(H=900, W=1600, flip=None, scale=None)

In [19]:
print(resize, resize_dims, crop, flip, rotate)

0.44 (704, 396) (0, 140, 704, 396) False 0


In [5]:
multi_adj_frame_id_cfg = (1, 1+1, 1)

In [6]:
num_adj=len(range(*multi_adj_frame_id_cfg))

In [7]:
num_adj

1